In [32]:
# ============================================================
# CLASSIFICATION ASSIGNMENT - Framingham Heart Study
# ============================================================

# STEP 1: Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# Classification Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# For handling imbalanced data
from imblearn.over_sampling import SMOTE

print("✅ Step 1: Libraries Imported")

# ============================================================
# STEP 2: Load & Fix Dataset
# ============================================================
# Define column names based on standard Framingham dataset structure
columns = [
    'male', 'age', 'education', 'currentSmoker', 'cigsPerDay', 
    'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes', 
    'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose', 'TenYearCHD'
]

try:
    # Read file without header, assign names, and treat 'NA' as missing values
    df = pd.read_csv('framingham.csv', header=None, names=columns, na_values=['NA'])
    
    print(f"✅ Step 2: Dataset Loaded. Shape: {df.shape}")
    print("\nFirst 5 Rows:")
    display(df.head())

except FileNotFoundError:
    print("❌ ERROR: 'framingham.csv' not found.")
    exit()

# ============================================================
# STEP 3: Data Preprocessing
# ============================================================
# Check for missing values
print("\n🔍 Missing Values Check:")
print(df.isnull().sum())

# Handle Missing Values: Fill numeric columns with Median
df.fillna(df.median(), inplace=True)

# Verify no missing values remain
print(f"\nMissing values after handling: {df.isnull().sum().sum()}")

# ============================================================
# STEP 4: Separate Features (X) and Target (y)
# ============================================================
X = df.drop('TenYearCHD', axis=1)
y = df['TenYearCHD']

print(f"\n📊 Target Distribution (Before Balancing):")
print(y.value_counts())

# Split into Training and Testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# ============================================================
# STEP 5: Feature Scaling & Handling Imbalance
# ============================================================
# Scale features (Important for SVM, KNN, Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Handle Imbalanced Data using SMOTE (Synthetic Minority Over-sampling Technique)
# Only apply SMOTE to the Training set to avoid data leakage
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print(f"\n📊 Target Distribution (After SMOTE):")
print(pd.Series(y_train_balanced).value_counts())

# ============================================================
# STEP 6: Train & Compare Classification Models
# ============================================================
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results = []

print("\n" + "="*70)
print(f"{'Model':<25} | {'Accuracy':<10} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10}")
print("="*70)

for name, model in models.items():
    # Train model on BALANCED data
    model.fit(X_train_balanced, y_train_balanced)
    
    # Predict on ORIGINAL scaled test data (to see real world performance)
    y_pred = model.predict(X_test_scaled)
    
    # Calculate Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results.append({'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1})
    print(f"{name:<25} | {acc:<10.4f} | {prec:<10.4f} | {rec:<10.4f} | {f1:<10.4f}")

print("="*70)

# ============================================================
# STEP 7: Detailed Report for Best Model (Random Forest)
# ============================================================
# Let's pick Random Forest for detailed analysis as it usually performs well
best_model = models['Random Forest']
y_pred_best = best_model.predict(X_test_scaled)

print("\n📄 CLASSIFICATION REPORT (Random Forest):")
print(classification_report(y_test, y_pred_best))

print("\n CONFUSION MATRIX:")
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix - Random Forest')
plt.show()

# ============================================================
# STEP 8: Hyperparameter Tuning (Random Forest)
# ============================================================
print("\n🔧 Step 8: Tuning Random Forest...")

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}

# Use GridSearchCV with cross-validation
# Note: We use the balanced training data for tuning
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='f1', # Optimize for F1 score because data was imbalanced
    n_jobs=-1
)

grid_search.fit(X_train_balanced, y_train_balanced)

print(f"   Best Parameters: {grid_search.best_params_}")

# Evaluate Tuned Model
tuned_model = grid_search.best_estimator_
y_pred_tuned = tuned_model.predict(X_test_scaled)

print("\n🏆 TUNED MODEL PERFORMANCE:")
print(f"   Accuracy  : {accuracy_score(y_test, y_pred_tuned):.4f}")
print(f"   Precision : {precision_score(y_test, y_pred_tuned):.4f}")
print(f"   Recall    : {recall_score(y_test, y_pred_tuned):.4f}")
print(f"   F1-Score  : {f1_score(y_test, y_pred_tuned):.4f}")

print("\n✅ Assignment Completed Successfully!")

✅ Step 1: Libraries Imported
✅ Step 2: Dataset Loaded. Shape: (4239, 16)

First 5 Rows:


,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
1,1,39,4,0,0,0,0,0,0,195,106,70,26.97,80,77,0
2,0,46,2,0,0,0,0,0,0,250,121,81,28.73,95,76,0
3,1,48,1,1,20,0,0,0,0,245,127.5,80,25.34,75,70,0
4,0,61,3,1,30,0,0,1,0,225,150,95,28.58,65,103,1



🔍 Missing Values Check:
male                 0
age                  0
education          105
currentSmoker        0
cigsPerDay          29
BPMeds              53
prevalentStroke      0
prevalentHyp         0
diabetes             0
totChol             50
sysBP                0
diaBP                0
BMI                 19
heartRate            1
glucose            388
TenYearCHD           0
dtype: int64


TypeError: Cannot convert [['male' '1' '0' ... '0' '0' '0']
 ['age' '39' '46' ... '48' '44' '52']
 ['education' '4' '2' ... '2' '1' '2']
 ...
 ['heartRate' '80' '95' ... '84' '86' '80']
 ['glucose' '77' '76' ... '86' nan '107']
 ['TenYearCHD' '0' '0' ... '0' '0' '0']] to numeric